# 07. 사람 라벨로 모델 검증: 일치도(κ)와 정확도(F1) (2026-10-07)

**목적**: 06 노트북의 결과(급증 구간에 분노 증가)를 믿을 수 있는지 확인한다. 팀원 2명(세은 = A, 다빈 = B)이 같은 리뷰 50건에 라벨링 지침대로 감정과 원인 구절을 붙였고, 이걸로 두 가지를 계산한다.

| 질문 | 지표 | 논문 값 (Attri et al., 2025) |
|---|---|---|
| 사람끼리 얼마나 일치하나? (과제가 명확한가) | 감정: Cohen κ / Fleiss κ, 원인 구절: 단어(토큰) 단위 κ | 감정 κ 0.88, trigger κ 0.84 (Table 1) |
| 모델이 사람과 얼마나 맞나? | 감정: 정밀도(P)·재현율(R)·F1, 원인 구절: ROUGE-L | Mistral-7B + EOT-DETECT 감정 F1 0.86, trigger ROUGE-L 0.68 (Table 2·3) |

**입력** (`data/processed/eot_pilot/`): `label_sheet_A.csv`, `label_sheet_B.csv`, `eot_emotions.csv`, `eot_triggers.csv`, `eot_raw.jsonl`

## 1. 준비와 라벨 시트 읽기

시트 규칙 (라벨링 지침 6장): 감정 칸이 **비어 있지 않으면** 그 감정이 있음, 칸 안의 원인 구절은 ` | `로 구분, 8개 감정이 모두 비면 Neutral.
- 엑셀이 저장하면서 넣는 앞뒤 공백, `nan` 같은 값은 빈칸으로 본다.
- Neutral 칸은 사람이 실수로 안 채웠어도 **8개 감정이 모두 비면 자동으로 Neutral**로 본다. 반대로 감정이 있는데 Neutral = 1이면 감정을 우선한다.

In [1]:
%pip install -q scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

D = Path("../data/processed/eot_pilot")
EMO8 = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation"]
EMO9 = EMO8 + ["Neutral"]

def read_sheet(path):
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    lab, trg = {}, []
    for r in df.itertuples(index=False):
        r = r._asdict()
        rid, text = r["review_id"].strip(), r["text"]
        row = {}
        for e in EMO8:
            cell = str(r.get(e, "")).strip()
            cell = "" if cell.lower() in ("nan", "none", "0") else cell
            row[e] = int(cell != "")
            for t in [x.strip() for x in cell.split("|") if x.strip()]:
                trg.append({"review_id": rid, "emotion": e, "trigger": t, "exact": t in text})
        row["Neutral"] = int(sum(row.values()) == 0)
        lab[rid] = row
    return pd.DataFrame.from_dict(lab, orient="index")[EMO9], pd.DataFrame(trg, columns=["review_id", "emotion", "trigger", "exact"]), df

A, trA, sheetA = read_sheet(D / "label_sheet_A.csv")
B, trB, sheetB = read_sheet(D / "label_sheet_B.csv")
TEXT = dict(zip(sheetA["review_id"].str.strip(), sheetA["text"]))
assert list(A.index) == list(B.index), "두 시트의 리뷰 순서·ID가 다르다"
print("라벨 리뷰 수:", len(A))
pd.DataFrame({"A(세은) 감정 수": A.sum(), "B(다빈) 감정 수": B.sum()})

라벨 리뷰 수: 50


,A(세은) 감정 수,B(다빈) 감정 수
Joy,25,25
Trust,18,14
Fear,3,2
Surprise,3,4
Sadness,12,11
Disgust,13,15
Anger,20,21
Anticipation,9,13
Neutral,1,3


## 2. [점검] 사람이 적은 원인 구절이 본문에 그대로 있나

라벨링 지침대로 본문에서 복사했다면 모두 `exact = True`여야 한다. False가 있으면 직접 타이핑했거나 엑셀이 글자를 바꾼 것이다. 아래 표에 나오는 구절은 해당 라벨러에게 확인을 부탁한다. (κ와 감정 F1 계산에는 영향이 없고, trigger 비교에만 영향이 있다.)

In [3]:
for name, tr in [("A(세은)", trA), ("B(다빈)", trB)]:
    print(f"{name}: 구절 {len(tr)}개, 본문에 그대로 있음 {int(tr['exact'].sum())}개")
pd.concat([trA.assign(라벨러="A"), trB.assign(라벨러="B")]).query("not exact")[["라벨러", "review_id", "emotion", "trigger"]]

A(세은): 구절 103개, 본문에 그대로 있음 101개
B(다빈): 구절 105개, 본문에 그대로 있음 101개


,라벨러,review_id,emotion,trigger
15,A,2dd41c91f5710d07,Surprise,"12 big rolls' = 20 regular Would like to see the visual math on that and show a regular roll next to one of these ""b..."
16,A,2dd41c91f5710d07,Anger,"12 big rolls' = 20 regular Would like to see the visual math on that and show a regular roll next to one of these ""b..."
1,B,979fbc935ffb293a,Anger,"Very tough, rough texture"
17,B,5a128bfa0b1e19cf,Joy,Lucky. Great tp.
20,B,2dd41c91f5710d07,Surprise,"12 big rolls' = 20 regular Would like to see the visual math on that and show a regular roll next to one of these ""b..."
43,B,dfea98f3ee2ecf27,Joy,loves this tissue. Enjoying having it


## 3. 사람끼리 일치도 (감정)

- **Cohen κ**: 두 사람이 우연히 일치할 확률을 뺀 일치도. 1이면 완전 일치, 0이면 우연 수준. 0.81 이상을 "거의 완전한 일치"로 본다 (Landis & Koch, 1977; 논문도 같은 기준).
- 감정별 κ와, 50건 × 9감정 = 450개 판단을 한꺼번에 본 **전체 κ**를 계산한다.
- **Fleiss κ**: 논문이 쓴 지표 (평가자 3명). 2명일 때도 계산할 수 있어서 비교용으로 같이 낸다.
- 건수가 너무 적은 감정(둘 다 거의 안 씀)은 κ가 불안정하다. 그래서 감정별 건수를 같이 본다.

In [4]:
def cohen_kappa(x, y):
    x, y = np.asarray(x), np.asarray(y)
    po = (x == y).mean()
    pe = x.mean() * y.mean() + (1 - x.mean()) * (1 - y.mean())
    return np.nan if pe == 1 else (po - pe) / (1 - pe)

def fleiss_kappa_2(x, y):
    x, y = np.asarray(x), np.asarray(y)
    po = (x == y).mean()
    p1 = (x.mean() + y.mean()) / 2
    pe = p1**2 + (1 - p1)**2
    return np.nan if pe == 1 else (po - pe) / (1 - pe)

rows = [{"감정": e, "A_건수": int(A[e].sum()), "B_건수": int(B[e].sum()),
         "일치율": (A[e] == B[e]).mean(), "Cohen_κ": cohen_kappa(A[e], B[e])} for e in EMO9]
kap = pd.DataFrame(rows)
allA, allB = A[EMO9].values.ravel(), B[EMO9].values.ravel()
overall = {"전체 Cohen κ": cohen_kappa(allA, allB), "전체 Fleiss κ": fleiss_kappa_2(allA, allB), "전체 일치율": (allA == allB).mean()}
print({k: round(v, 3) for k, v in overall.items()})
kap.round(3)

{'전체 Cohen κ': np.float64(0.877), '전체 Fleiss κ': np.float64(0.877), '전체 일치율': np.float64(0.956)}


,감정,A_건수,B_건수,일치율,Cohen_κ
0,Joy,25,25,1.00,1.000
1,Trust,18,14,0.92,0.818
2,Fear,3,2,0.98,0.790
3,Surprise,3,4,0.98,0.847
4,Sadness,12,11,0.98,0.944
5,Disgust,13,15,0.96,0.901
6,Anger,20,21,0.94,0.876
7,Anticipation,9,13,0.88,0.654
8,Neutral,1,3,0.96,0.485


## 4. 사람끼리 일치도 (원인 구절, 토큰 단위)

논문 방식 (5.1절): 원인 구절을 **단어(토큰) 단위**로 나눠서, 리뷰의 각 단어가 "원인 구절 안에 있다/없다"를 두 사람이 똑같이 판단했는지 κ로 본다. 그래서 "very happy"와 "happy"처럼 경계만 다른 경우도 부분 일치로 잡힌다.
- 리뷰 본문을 공백으로 나눈 단어마다, 그 사람이 적은 구절(어느 감정이든) 안에 들어가면 1.
- 본문에 그대로 있는 구절(`exact`)만 쓴다. 위치를 찾을 수 없기 때문이다.

In [5]:
def token_mask(text, triggers):
    spans = []
    for t in triggers:
        i = text.find(t)
        if i >= 0:
            spans.append((i, i + len(t)))
    mask = []
    for m in re.finditer(r"\S+", text):
        mask.append(int(any(m.start() < b and m.end() > a for a, b in spans)))
    return mask

ta, tb = [], []
for rid in A.index:
    text = TEXT[rid]
    ta += token_mask(text, trA.query("review_id == @rid and exact")["trigger"])
    tb += token_mask(text, trB.query("review_id == @rid and exact")["trigger"])
trigger_kappa = cohen_kappa(ta, tb)
print(f"원인 구절 토큰 단위 Cohen κ = {trigger_kappa:.3f} (토큰 {len(ta)}개, A 표시 {sum(ta)}, B 표시 {sum(tb)})")

원인 구절 토큰 단위 Cohen κ = 0.791 (토큰 1463개, A 표시 1040, B 표시 913)


## 5. 정답(gold) 만들기

논문은 전문가 3명의 라벨을 모아 정답을 만들었다. 우리는 2명이라 **두 사람이 모두 표시한 감정만 정답**으로 본다 (2명 중 과반 = 2명 모두). 원인 구절은 두 사람 것을 모두 남기되, 겹치면 **더 긴 구간**을 남긴다 (논문의 longest-span 규칙, p.5520).
- 두 사람이 다르게 판단한 리뷰는 6절 표에서 따로 본다. 나중에 둘이 상의해서 정답을 고치면 그 결과를 `label_gold.csv`로 저장해 이 셀을 바꿔 쓸 수 있다.
- 참고로 모델 정확도는 A, B 각각을 정답으로 놓고도 계산한다. 사람 B를 정답으로 놓고 사람 A의 F1을 구하면 **"사람 수준"** 이 어느 정도인지 알 수 있다.

In [6]:
gold = (A[EMO8] & B[EMO8]).astype(int)
gold["Neutral"] = (gold[EMO8].sum(axis=1) == 0).astype(int)

def longest_spans(trigs):
    trigs = sorted(set(trigs), key=len, reverse=True)
    keep = []
    for t in trigs:
        if not any(t in k for k in keep):
            keep.append(t)
    return keep

gold_trig = {}
for rid in gold.index:
    for e in EMO8:
        if gold.loc[rid, e]:
            ts = list(trA.query("review_id == @rid and emotion == @e and exact")["trigger"]) + \
                 list(trB.query("review_id == @rid and emotion == @e and exact")["trigger"])
            gold_trig[(rid, e)] = longest_spans(ts)
print("정답 감정 수:", gold.sum().to_dict())

정답 감정 수: {'Joy': 25, 'Trust': 14, 'Fear': 2, 'Surprise': 3, 'Sadness': 11, 'Disgust': 13, 'Anger': 19, 'Anticipation': 8, 'Neutral': 3}


## 6. [점검] 모델 정확도 (감정)

- 모델 결과(`eot_emotions.csv`)에서 라벨 50건만 꺼낸다.
- **정밀도(P)**: 모델이 "있다"고 한 감정 중 정답인 비율. **재현율(R)**: 정답 감정 중 모델이 찾은 비율. **F1**: 둘의 조화평균.
- **micro**: 50건 × 9감정의 판단을 모두 합쳐 계산 (건수가 많은 Joy 영향이 큼). **macro**: 감정별 F1의 평균 (드문 감정도 똑같이 반영). 논문 표는 감정 P/R/F1을 하나로 보고하므로 micro를 주 지표로 쓴다.
- **보조**: 모델이 쓴 "Disappointment"를 Sadness로 바꿨을 때 (06 노트북과 같은 보조 분석).

In [7]:
me = pd.read_csv(D / "eot_emotions.csv", dtype={"review_id": str}).set_index("review_id")
raw = pd.read_json(D / "eot_raw.jsonl", lines=True, dtype={"review_id": str})
M = me.loc[gold.index, EMO9].fillna(0).astype(int)
dis_ids = set(raw.loc[raw["output"].str.contains(r'"emotion"\s*:\s*"[^"]*disappoint', case=False, regex=True), "review_id"])
M2 = M.copy(); M2.loc[M2.index.isin(dis_ids), "Sadness"] = 1
M2["Neutral"] = ((M2[EMO8].sum(axis=1) == 0)).astype(int)

def prf(pred, true, labels=EMO9):
    p, t = pred[labels].values.ravel(), true[labels].values.ravel()
    tp, fp, fn = ((p == 1) & (t == 1)).sum(), ((p == 1) & (t == 0)).sum(), ((p == 0) & (t == 1)).sum()
    P = tp / (tp + fp) if tp + fp else np.nan; R = tp / (tp + fn) if tp + fn else np.nan
    F = 2 * P * R / (P + R) if P and R else 0.0
    return P, R, F

def macro_f1(pred, true, labels=EMO9):
    fs = [prf(pred, true, [e])[2] for e in labels if true[e].sum() + pred[e].sum() > 0]
    return np.mean(fs)

rows = []
for name, pred, true in [("모델 vs 정답(둘 다 표시)", M, gold), ("모델 vs A(세은)", M, A), ("모델 vs B(다빈)", M, B),
                         ("모델(실망→슬픔) vs 정답", M2, gold), ("사람 A vs 사람 B (사람 수준)", A, B)]:
    P, R, F = prf(pred, true)
    rows.append({"비교": name, "P": P, "R": R, "micro_F1": F, "macro_F1": macro_f1(pred, true)})
acc = pd.DataFrame(rows)
acc.round(3)

,비교,P,R,micro_F1,macro_F1
0,모델 vs 정답(둘 다 표시),0.805,0.337,0.475,0.285
1,모델 vs A(세은),0.829,0.327,0.469,0.294
2,모델 vs B(다빈),0.805,0.306,0.443,0.265
3,모델(실망→슬픔) vs 정답,0.722,0.398,0.513,0.328
4,사람 A vs 사람 B (사람 수준),0.923,0.889,0.906,0.841


In [8]:
per = pd.DataFrame([{"감정": e, "정답_건수": int(gold[e].sum()), "모델_건수": int(M[e].sum()),
                     "P": prf(M, gold, [e])[0], "R": prf(M, gold, [e])[1], "F1": prf(M, gold, [e])[2]} for e in EMO9])
per.round(2)

,감정,정답_건수,모델_건수,P,R,F1
0,Joy,25,24,0.88,0.84,0.86
1,Trust,14,1,1.00,0.07,0.13
2,Fear,2,0,NaN,0.00,0.00
3,Surprise,3,2,0.50,0.33,0.40
4,Sadness,11,2,1.00,0.18,0.31
5,Disgust,13,4,0.75,0.23,0.35
6,Anger,19,4,1.00,0.21,0.35
7,Anticipation,8,4,0.25,0.12,0.17
8,Neutral,3,0,NaN,0.00,0.00


## 7. 모델 정확도 (원인 구절, ROUGE-L)

- 정답과 모델이 **둘 다 같은 감정을 찾은 경우**에, 정답 구절마다 모델 구절 중 가장 비슷한 것과의 **ROUGE-L F1**(가장 긴 공통 단어 순서 기준 겹침)을 구해 평균한다. 논문 Table 3의 trigger 지표다.
- 대소문자는 무시한다.

In [9]:
def rouge_l(a, b):
    x, y = a.lower().split(), b.lower().split()
    if not x or not y:
        return 0.0
    dp = [[0] * (len(y) + 1) for _ in range(len(x) + 1)]
    for i in range(len(x)):
        for j in range(len(y)):
            dp[i + 1][j + 1] = dp[i][j] + 1 if x[i] == y[j] else max(dp[i][j + 1], dp[i + 1][j])
    l = dp[-1][-1]
    if l == 0:
        return 0.0
    p, r = l / len(y), l / len(x)
    return 2 * p * r / (p + r)

mt = pd.read_csv(D / "eot_triggers.csv", dtype={"review_id": str})
scores = []
for (rid, e), gts in gold_trig.items():
    preds = list(mt.query("review_id == @rid and emotion == @e")["trigger"])
    if not preds or not gts:
        continue
    scores += [max(rouge_l(g, p) for p in preds) for g in gts]
trigger_rouge = np.mean(scores) if scores else np.nan
print(f"원인 구절 ROUGE-L = {trigger_rouge:.3f} (정답 구절 {len(scores)}개 기준)")

원인 구절 ROUGE-L = 0.585 (정답 구절 33개 기준)


## 8. [점검] 어디서 틀렸나

정답과 모델이 다른 리뷰를 본다. 논문이 말한 오류 유형(p.5522)이 그대로 나오는지 확인한다: **Joy와 Trust 혼동**, 비꼬기 같은 **암시적 감정 놓침**, 감정 하나에 원인이 여러 개일 때 **두 번째 원인 놓침**.

In [10]:
diff = []
for rid in gold.index:
    g = [e for e in EMO9 if gold.loc[rid, e]]; m = [e for e in EMO9 if M.loc[rid, e]]
    if set(g) != set(m):
        diff.append({"review_id": rid, "정답": ", ".join(g), "모델": ", ".join(m),
                     "A": ", ".join(e for e in EMO9 if A.loc[rid, e]), "B": ", ".join(e for e in EMO9 if B.loc[rid, e]),
                     "본문": TEXT[rid][:150]})
print("정답과 다른 리뷰:", len(diff), "/", len(gold))
pd.DataFrame(diff)

정답과 다른 리뷰: 45 / 50


,review_id,정답,모델,A,B,본문
0,979fbc935ffb293a,"Disgust, Anger",,"Disgust, Anger","Disgust, Anger","Very tough on lady bits, rough texture. Sandpaper is the best description."
1,168697c86616b468,"Sadness, Anger",Sadness,"Sadness, Anger","Sadness, Disgust, Anger",This is not the same quality as the Angel Soft double rolls I've used for years. :(
2,1186ebdb43a92c80,"Disgust, Anger",,"Disgust, Anger","Disgust, Anger","Incredibly thin, one ply tp. Similar to Scotts. Not worth it - I’m going back to Chairman which costs more but we us..."
3,d90d643583e71876,"Joy, Trust, Anticipation",Joy,"Joy, Trust, Anticipation","Joy, Trust, Surprise, Anticipation",I like that they’re roll free. Better for the environment! Scott’s makes good tp.
4,3b5a7a7566973e8d,"Joy, Trust",Joy,"Joy, Trust","Joy, Trust, Anticipation",Wonderful buy when you have a big family. It's hassle free to have your household products delivered right at your f...
5,bbbc6faf2c2812fb,"Joy, Trust",Joy,"Joy, Trust","Joy, Trust, Anticipation","My favorite toilet paper. We kept having a few people saying they loved our toilet paper when they came, so for Chri..."
6,5a128bfa0b1e19cf,"Joy, Trust",Joy,"Joy, Trust","Joy, Trust, Anticipation","Lucky I got the product, should have ordered more. Great tp."
7,2dd41c91f5710d07,"Surprise, Anger",,"Surprise, Anger","Surprise, Anger","""12 big rolls' = 20 regular"" Would like to see the visual math on that and show a regular roll next to one of these ..."
8,4f216d930c8c2405,"Sadness, Anger",,"Sadness, Anger","Sadness, Disgust, Anger",This is not the big rolls of toilet paper I thought I was getting...I'm so disappointed .I could have bought these l...
9,e832df6dfed1e088,"Disgust, Anger",Anger,"Disgust, Anger","Disgust, Anger",Price gouging at its finest. Four rolls of toilet paper for $35? I don't think so.


## 9. 로그에 붙일 문장

아래 출력을 복사해 `logs/preprocessing_log.md` 맨 아래에 붙인다.

In [11]:
g = acc.set_index("비교")
lines = [
    "### 2026-10-07 T5. 사람 라벨 50건으로 모델 검증 (김지우)",
    "노트북: `notebooks/07_eot_validation.ipynb`. 라벨러: 정세은(A), 김다빈(B), 지침 `docs/EOT_라벨링_지침.md`",
    "",
    f"- 사람끼리 일치도: 감정 Cohen κ {overall['전체 Cohen κ']:.2f} (Fleiss κ {overall['전체 Fleiss κ']:.2f}), 원인 구절 토큰 κ {trigger_kappa:.2f} / 논문 0.88, 0.84",
    f"- 모델 vs 정답(두 사람 모두 표시한 감정): P {g.loc['모델 vs 정답(둘 다 표시)', 'P']:.2f} / R {g.loc['모델 vs 정답(둘 다 표시)', 'R']:.2f} / micro F1 {g.loc['모델 vs 정답(둘 다 표시)', 'micro_F1']:.2f} (macro {g.loc['모델 vs 정답(둘 다 표시)', 'macro_F1']:.2f}) / 논문 F1 0.86",
    f"- 사람 A vs 사람 B micro F1 {g.loc['사람 A vs 사람 B (사람 수준)', 'micro_F1']:.2f} (사람 수준 참고값)",
    f"- 실망→슬픔 보조: micro F1 {g.loc['모델(실망→슬픔) vs 정답', 'micro_F1']:.2f}",
    f"- 원인 구절 ROUGE-L {trigger_rouge:.2f} / 논문 0.68",
    f"- 정답과 다른 리뷰 {len(diff)}/{len(gold)}건",
]
print("\n".join(lines))

### 2026-10-07 T5. 사람 라벨 50건으로 모델 검증 (김지우)
노트북: `notebooks/07_eot_validation.ipynb`. 라벨러: 정세은(A), 김다빈(B), 지침 `docs/EOT_라벨링_지침.md`

- 사람끼리 일치도: 감정 Cohen κ 0.88 (Fleiss κ 0.88), 원인 구절 토큰 κ 0.79 / 논문 0.88, 0.84
- 모델 vs 정답(두 사람 모두 표시한 감정): P 0.80 / R 0.34 / micro F1 0.47 (macro 0.29) / 논문 F1 0.86
- 사람 A vs 사람 B micro F1 0.91 (사람 수준 참고값)
- 실망→슬픔 보조: micro F1 0.51
- 원인 구절 ROUGE-L 0.58 / 논문 0.68
- 정답과 다른 리뷰 45/50건
